# Processing and Cleaning ClinVar and dbNSFP data

In [5]:
import os
import subprocess
import polars as pl
import time
import re

## 1. ClinVar 2025-01

Use ```../1_inputs/download_data.sh``` to download correct version of ClinVar data from https://ftp.ncbi.nlm.nih.gov/pub/clinvar/tab_delimited/archive/variant_summary_2025-01.txt.gz

In [7]:
dest_path = "../1_inputs/raw_inputs/clinvar/variant_summary_2025-01.txt.gz"
## Define the new processed and cleaned clinvar file
processed_clinvar_parq = "../3_outputs/processed_data/clinvar/clinvar_processed.parquet"

In [9]:
pl.enable_string_cache()

## Read CSV with Polars (auto-decompresses .gz)
clinvar = pl.read_csv(dest_path, separator='\t', infer_schema_length=1000000)
## Process data with Polars expressions
clinvar = (
    clinvar
    .filter(pl.col("Assembly") == "GRCh38")
    .with_columns(
        pl.col("Name")
        # Split the string on '.', '(' or ')'
        .map_elements(lambda s: re.split(r"[.\(\)]", s), pl.List(pl.String))
        .alias("Name_listed")
    )
)
clinvar = clinvar.with_columns(
    pl.col("Name_listed").list.get(0, null_on_oob=True).alias("NM"),
    pl.col("Name_listed").list.get(1, null_on_oob=True).alias("version"), 
    pl.col("Name_listed").list.get(2, null_on_oob=True).alias("gene"),
    pl.col("Name_listed").list.get(3, null_on_oob=True).alias("c"),
    pl.col("Name_listed").list.get(4, null_on_oob=True).alias("clinvar_nt_change"),
    pl.col("Name_listed").list.get(5, null_on_oob=True).alias("p"),
    pl.col("Name_listed").list.get(6, null_on_oob=True).alias("clinvar_aa_change")
).with_columns(
        SPDI=pl.col("ChromosomeAccession") + ":" + 
            pl.col("PositionVCF").cast(str) + ":" +
            pl.col("ReferenceAlleleVCF") + ":" +
            pl.col("AlternateAlleleVCF"),
        RefSeq_mRNA=pl.col("NM") + "." + pl.col("version"),
        StarStatus=pl.when(pl.col("ReviewStatus").str.contains(r'single|conflicting'))
            .then(1)
            .when(pl.col("ReviewStatus").str.contains('multiple'))
            .then(2)
            .when(pl.col("ReviewStatus").str.contains('expert'))
            .then(3)
            .when(pl.col("ReviewStatus").str.contains('practice'))
            .then(4)
            .otherwise(None)
)

## Create the mapping expression with precise logic for a cleaner ClinVar Annotation
clinvar_mapping = (
    # First check for Conflicting interpretations (highest priority)
    pl.when(pl.col("ClinicalSignificance").str.to_lowercase().str.contains("conflicting"))
    .then(pl.lit("Conflicting"))
    # Then check for Uncertain significance (VUS)
    .when(pl.col("ClinicalSignificance").str.to_lowercase().str.contains("uncertain"))
    .then(pl.lit("VUS"))
    # Check for Pathogenic
    .when(pl.col("ClinicalSignificance").str.to_lowercase().str.contains("pathogenic"))
    .then(pl.lit("Pathogenic"))
    # Check for Benign
    .when(pl.col("ClinicalSignificance").str.to_lowercase().str.contains("benign"))
    .then(pl.lit("Benign"))
    # Everything else goes to Others
    .otherwise(pl.lit("Others"))
)

## Create the mapping expression with precise logic for a cleaner ClinVar Annotation USING Pillar team's strict Clinical Significance priority
clinvar_mapping_strict_pp = (
    # Pathogenic / Likely pathogenic
    pl.when(pl.col("ClinicalSignificance")=="Pathogenic")
    .then(pl.lit("Pathogenic"))
    .when(pl.col("ClinicalSignificance")=="Pathogenic/Likely pathogenic")
    .then(pl.lit("Likely pathogenic"))
    .when(pl.col("ClinicalSignificance")=="Likely pathogenic")
    .then(pl.lit("Likely pathogenic"))

    # Benign / Likely benign
    .when(pl.col("ClinicalSignificance")=="Benign")
    .then(pl.lit("Benign"))
    .when(pl.col("ClinicalSignificance")=="Benign/Likely benign")
    .then(pl.lit("Likely benign"))
    .when(pl.col("ClinicalSignificance")=="Likely benign")
    .then(pl.lit("Likely benign"))
    
    # Then check for Uncertain significance (VUS)
    .when(pl.col("ClinicalSignificance")=="Uncertain significance")
    .then(pl.lit("VUS"))
    
    # Check for Conflicting
    .when(pl.col("ClinicalSignificance")=="Conflicting classifications of pathogenicity")
    .then(pl.lit("Conflicting"))

    # Everything else goes to Others
    .otherwise(pl.lit("Others"))
)

# Apply the mapping to create a new column
clinvar = clinvar.with_columns(
    clinvar_clnsig_clean=clinvar_mapping,
    clinvar_clnsig_clean_pp_strict=clinvar_mapping_strict_pp)

# Create a dataframe with unique mappings to check accuracy
# Sort by original value for easier verification
unique_mappings = clinvar.select(
    pl.col("ClinicalSignificance"),
    pl.col("clinvar_clnsig_clean"),
    pl.col("clinvar_clnsig_clean_pp_strict")
).unique().sort("ClinicalSignificance")

# Print each mapping to verify accuracy
print("Original Value -> Mapped Value -> Mapped Value (Pillar Team Strict)")
print("-" * 70)
for row in unique_mappings.rows():
    print(f"{row[0]} -> {row[1]} -> {row[2]} ")

## Make sure we only drop columns that actually exist
columns_to_drop = [col for col in ['NM', 'version', 'gene', 'c', 'p', 'ReferenceAllele', 'AlternateAllele', 'Name_listed'] 
                  if col in clinvar.columns]
if columns_to_drop:
    clinvar = clinvar.drop(columns_to_drop)

# Now cast to categorical
clinvar = clinvar.with_columns(
    pl.col(["Assembly", "ClinicalSignificance", "ReviewStatus", "clinvar_clnsig_clean", "clinvar_clnsig_clean_pp_strict"]).cast(pl.Categorical)
)

Original Value -> Mapped Value -> Mapped Value (Pillar Team Strict)
----------------------------------------------------------------------
- -> Others -> Others 
Affects -> Others -> Others 
Affects; risk factor -> Others -> Others 
Benign -> Benign -> Benign 
Benign/Likely benign -> Benign -> Likely benign 
Benign/Likely benign; association -> Benign -> Others 
Benign/Likely benign; drug response -> Benign -> Others 
Benign/Likely benign; drug response; other -> Benign -> Others 
Benign/Likely benign; other -> Benign -> Others 
Benign/Likely benign; other; risk factor -> Benign -> Others 
Benign/Likely benign; risk factor -> Benign -> Others 
Benign; Affects -> Benign -> Others 
Benign; Affects; association; other -> Benign -> Others 
Benign; association -> Benign -> Others 
Benign; confers sensitivity -> Benign -> Others 
Benign; drug response -> Benign -> Others 
Benign; other -> Benign -> Others 
Benign; protective -> Benign -> Others 
Benign; risk factor -> Benign -> Others 
Confl

In [ ]:
# Write to Parquet with ZSTD compression
# clinvar.write_parquet(processed_clinvar_parq, 
#                      compression="zstd", 
#                      statistics=True)

In [ ]:
processed_clinvar_parq = "../3_outputs/processed_data/clinvar/clinvar_processed.parquet"
clinvar = pl.scan_parquet(processed_clinvar_parq)
clinvar_snv = clinvar.filter(pl.col("Type")=='single nucleotide variant') ##.select(pl.col("Type")).unique().collect().to_numpy()
clinvar_snv.collect().shape

## 2. dbNSFP v5.0a

Using ```../1_inputs/download_data.sh``` to download data from https://drive.google.com/file/d/1HNoZUswadC1ODavU7AUvm0DZ13jKVFKW/view

We also use dbNSFP for gnomad allele frequency as well, which used gnomadv4.1 for annotations. Documentation at: https://www.dbnsfp.org/releases

In [ ]:
# def convert_to_parquet(chr_num):
#     file_path = f"../1_inputs/raw_inputs/dbNSFP5.0a/dbNSFP5.0a_variant.chr{chr_num}.gz"
#     df = pl.read_csv(file_path, 
#                      separator='\t',
#                      n_threads=16,
#                      infer_schema_length=1000000,
#                      ignore_errors=True,
#                      truncate_ragged_lines=True,
#                      low_memory=False)
#     df = df.with_columns(
#         pl.col("aaref").replace(aa_single_to_three).alias("aaref_3"),
#         pl.col("aaalt").replace(aa_single_to_three).alias("aaalt_3"),
#     )
#     parquet_path = f"../1_inputs/raw_inputs/dbNSFP5.0a/dbNSFP5.0a_variant.chr{chr_num}.parquet"
#     df.write_parquet(parquet_path)
#     return parquet_path

# chr_num_list = [chr_file.split('.')[-2].split('chr')[-1] for chr_file in os.listdir('../1_inputs/raw_inputs/dbNSFP5.0a') if "chr" in chr_file and chr_file.endswith(".gz")]
# # Use ThreadPoolExecutor for parallel processing
# with concurrent.futures.ThreadPoolExecutor() as executor:
#     futures = [executor.submit(convert_to_parquet, chr_num) for chr_num in chr_num_list]
#     # Collect the results
#     parquet_files = [future.result() for future in concurrent.futures.as_completed(futures)]

# print("Conversion to Parquet completed.")

### Merge dbNSFP and ClinVar

dbNSFP and ClinVar datasets were merged together using ```VariantionID``` (ClinVar) and ```clinvar_id``` (dbNSFP), which are the unique identifier for each variant.

In [ ]:
# %%time

# with pl.StringCache():
#     chr_num_list = [chr_file.split('.')[-2].split('chr')[-1] for chr_file in os.listdir('../1_inputs/raw_inputs/dbNSFP5.0a/') if "chr" in chr_file and chr_file.endswith(".gz")]
#     clin_var_dfs = []
#     for chr_num in chr_num_list:
#         df_chr_par = pl.scan_parquet(f"../1_inputs/raw_inputs/dbNSFP5.0a/dbNSFP5.0a_variant.chr{chr_num}.parquet")
#         df_chr_par_clinvar = df_chr_par.filter(pl.col("clinvar_id")!='.')
#         clinvar_per_chr = clinvar_snv.filter(pl.col("Chromosome")==chr_num)#.collect()
#         clinvar_per_chr = clinvar_per_chr.with_columns(pl.col("VariationID").cast(pl.String).alias("clinvar_id"))
#         df_chr_par_clinvar = df_chr_par_clinvar.join(clinvar_per_chr, on="clinvar_id", how="left")
#         df_chr_par_clinvar = df_chr_par_clinvar.collect()
#         clin_var_dfs.append(df_chr_par_clinvar)

#     df_clinvar = pl.concat(clin_var_dfs, how="vertical_relaxed")
#     parquet_path = f"../3_outputs/processed_data/dbnsfp/dbNSFP5.0a_variant.clin_var_re-annot.parquet"
#     df_clinvar.write_parquet(parquet_path)

CPU times: user 45min 44s, sys: 2h 53min 5s, total: 3h 38min 50s
Wall time: 2min 24s


In [15]:
df_clinvar = pl.scan_parquet("../3_outputs/processed_data/dbnsfp/dbNSFP5.0a_variant.clin_var_re-annot.parquet")
# For testing purposes, limit to a small subset:
df_clin_subset = df_clinvar.limit(10)
# Collect the subset into an eager DataFrame for inspection
df_clin_subset = df_clin_subset.collect()

# Automatically detect columns with semicolon-separated patterns.
# We only check columns of Utf8 (string) type.
pattern_cols = [
    col for col in df_clinvar.collect_schema() 
    if df_clin_subset[col].dtype == pl.Utf8 and df_clin_subset[col].str.contains(";").any()
]
print("Columns containing ';':", pattern_cols)

df_clinvar_exploded = df_clinvar.with_columns(
    [pl.col(col).str.split(";").alias(col) for col in pattern_cols]
).explode(pattern_cols)

df_clinvar_exploded.collect().shape

Columns containing ';': ['aapos', 'genename', 'Ensembl_geneid', 'Ensembl_transcriptid', 'Ensembl_proteinid', 'Uniprot_acc', 'Uniprot_entry', 'HGVSc_snpEff', 'HGVSp_snpEff', 'HGVSc_VEP', 'HGVSp_VEP', 'APPRIS', 'GENCODE_basic', 'TSL', 'VEP_canonical', 'MANE', 'cds_strand', 'refcodon', 'codonpos', 'codon_degeneracy', 'SIFT_score', 'SIFT_pred', 'SIFT4G_score', 'SIFT4G_pred', 'Polyphen2_HDIV_score', 'Polyphen2_HDIV_pred', 'Polyphen2_HVAR_score', 'Polyphen2_HVAR_pred', 'MutationTaster_score', 'MutationTaster_pred', 'MutationTaster_model', 'MutationTaster_trees_benign', 'MutationTaster_trees_deleterious', 'MutationAssessor_score', 'MutationAssessor_pred', 'PROVEAN_score', 'PROVEAN_pred', 'VEST4_score', 'MetaRNN_score', 'MetaRNN_pred', 'REVEL_score', 'MutPred_score', 'MutPred_protID', 'MutPred_AAchange', 'MutPred_Top5features', 'MVP_score', 'gMVP_score', 'MPC_score', 'DEOGEN2_score', 'DEOGEN2_pred', 'LIST-S2_score', 'LIST-S2_pred', 'VARITY_R_score', 'VARITY_ER_score', 'VARITY_R_LOO_score', 'VA

(7475238, 427)

### De-duplicate them 

__By prioritizing the uniprot_id with a .pdb structure inferred by AF2__

In [17]:
def deduplicate_by_uniprot_lazy(ldf, preferred_uniprot_acc):
    # Create window specs for our operations
    window_spec = pl.col("clinvar_id")
    
    with pl.StringCache():
        # Add useful columns for filtering in a single pass:
        # 1. Count occurrences of each clinvar_id
        # 2. Check if Uniprot_acc is preferred
        # 3. Check if Uniprot_acc has no dash
        # 4. Create row number within each group to use for selection
        enhanced_ldf = ldf.with_columns([
            pl.col("clinvar_id").count().over(window_spec).alias("group_count"),
            pl.col("Uniprot_acc").is_in(preferred_uniprot_acc).alias("is_preferred"),
            (~pl.col("Uniprot_acc").str.contains("-")).alias("no_dash"),
            pl.col("Uniprot_acc").rank("dense").over(window_spec).alias("row_rank")
        ])
        
        # For non-duplicated entries - keep all
        non_duplicated = enhanced_ldf.filter(pl.col("group_count") == 1)
        
        # For duplicated entries, we'll create a priority column and then use that for selection
        duplicated = enhanced_ldf.filter(pl.col("group_count") > 1)

        # Create a priority column (1=preferred, 2=no dash, 3=other)
        prioritized = duplicated.with_columns([
            pl.when(pl.col("is_preferred"))
            .then(pl.lit(1))
            .when(pl.col("no_dash"))
            .then(pl.lit(2))
            .otherwise(pl.lit(3))
            .alias("priority")
        ])

        # Sort by priority and take the first record for each ID
        deduped = (
            prioritized
            .sort(["clinvar_id", "priority", "row_rank"])
            .group_by("clinvar_id")
            .agg(
                pl.all().exclude(["group_count", "is_preferred", "no_dash", "row_rank", "priority"]).first()
            )
        )

    deduped_df = pl.concat([non_duplicated.drop(["group_count", "is_preferred", "no_dash", "row_rank"]).select(pl.col(deduped.collect_schema().names())),
    deduped], how="vertical_relaxed")
    
    # Combine all results
    return deduped_df

uniprot_ids = [f.split('-')[1] for f in os.listdir("../1_inputs/raw_inputs/alphafold/UP000005640_9606_HUMAN_v4") if f.endswith("pdb.gz")]
## Let's prioritize the variants with PDB structures
df_clinvar_dedup_struct = deduplicate_by_uniprot_lazy(df_clinvar_exploded, uniprot_ids)
df_clinvar_dedup_struct.collect().shape

(1649580, 427)

In [ ]:
# with pl.StringCache():
#     df_clinvar_dedup_struct.collect().write_parquet(f"../3_outputs/processed_data/dbnsfp/dbNSFP5.0a_variant.clin_var_re-annot_pdb_dedup_variants.parquet")

## Do a test loading
df_clinvar_dedup_struct = pl.read_parquet(f"../3_outputs/processed_data/dbnsfp/dbNSFP5.0a_variant.clin_var_re-annot_pdb_dedup_variants.parquet")

## 3. AlphaFold2 Structures

### 3.1 EMBL AF accession

From accession_ids.csv from https://ftp.ebi.ac.uk/pub/databases/alphafold/README.txt:

2.) accession_ids.csv

This file contains a list of all the UniProt accessions that have predictions in
the current database version of AlphaFold DB. The file is in CSV format and
includes the following columns, separated by a comma:

* UniProt accession, e.g. A8H2R3
* First residue index (UniProt numbering), e.g. 1
* Last residue index (UniProt numbering), e.g. 199
* AlphaFold DB identifier, e.g. AF-A8H2R3-F1
* Latest version, e.g. 2

In [ ]:
remote_accid_csv = "../1_inputs/raw_inputs/alphafold/accession_ids.csv" ## from "https://ftp.ebi.ac.uk/pub/databases/alphafold/accession_ids.csv"
output_parquet = "../3_outputs/processed_data/alphafold/accession_id_mapping.parquet"
start_time = time.time()

accid_df = pl.scan_csv(
    remote_accid_csv,
    separator=',',
    has_header=False
)

accid_df = accid_df.rename({
    "column_1": "uniprot_id",
    "column_2": "first_res_idx",
    "column_3": "last_res_idx",
    "column_4": "af_db_identifier",
    "column_5": "latest_version"
})

print(accid_df.collect_schema().names())

### Add subfile index extraction (F1, F2 from AF-ID)
processed = accid_df.with_columns(
    pl.col("af_db_identifier")
    .str.split('-').arr.last()
    .str.extract(r"-F(\d+).*", 1)  # Capture all digits after F until end
    .alias("subfile_index")
).sort(["uniprot_id", "first_res_idx"])

# print(processed.collect())

# Sort by uniprot_id and first_res_idx for interval search
processed = processed.sort(["uniprot_id", "first_res_idx"])

# Write optimized Parquet with partitioning
# processed.collect().write_parquet(
#     output_parquet,
#     row_group_size=10000,
#     compression="zstd"
# )

end_time = time.time()
print(f"Processing completed in {end_time - start_time:.2f} seconds")

In [ ]:
accid_df = pl.scan_parquet(output_parquet)
accid_df = accid_df.collect()
accid_df["uniprot_id"].unique()

In [ ]:
# accid_df.filter(pl.col("subfile_index")!="1")
processed = accid_df.with_columns(
    pl.col("af_db_identifier")
    .str.split(by='-').list.last()
    .str.extract(r"F(\d+).*", 1)  # Capture all digits after F until end
    .alias("subfile_index")
).sort(["uniprot_id", "first_res_idx"])

processed.filter(~pl.col("af_db_identifier").str.ends_with("F1"))

In [ ]:
uniprot_ids = [f.split('-')[1] for f in os.listdir("../1_inputs/raw_inputs/alphafold/UP000005640_9606_HUMAN_v4") if f.endswith("pdb.gz")]
accid_df.filter(pl.col("uniprot_id").is_in(uniprot_ids)).unique(subset="uniprot_id")

### 3.2 Explore pLDDT and RSA

#### pLDDT

In [ ]:
import sys
import gzip
import glob
from Bio.PDB import PDBParser
from io import StringIO
from typing import Tuple, Optional

sys.path.insert(1, "data_processing_scripts")
import extract_plddt_scores as get_plddt
import calculate_rsa_scores as get_rsa

In [ ]:
pdb_file = "../1_inputs/raw_inputs/alphafold/UP000005640_9606_HUMAN_v4/AF-X6R8D5-F1-model_v4.pdb.gz"

# Step 3: Extract pLDDT Scores
def extract_plddt(pdb_file):
    """Parse AlphaFold output for per-residue pLDDT scores from a .gz PDB file"""
    with gzip.open(pdb_file, 'rt') as f:
        pdb_content = f.read()
    parser = PDBParser()
    structure = parser.get_structure("AF", StringIO(pdb_content))
    return [(residue.get_id()[1], residue.get_resname(), residue["CA"].bfactor) for residue in structure.get_residues()]

plddt_scores = extract_plddt(pdb_file=pdb_file)
plddt_scores[:10]

uniprot_id = "X6R8D5"
pdb_lines = []
with gzip.open(pdb_file, 'rt') as f:
    for i, line in enumerate(f):
        line = line[:-1].split()
        if line[0] == 'ATOM':
            if len(line) == 12: #solve issue for lines with merged columns
                pos = int(line[5])
                res = line[3]
                plddt = line[-2]
                row = [uniprot_id, pdb_file, pos, res, float(plddt)]
                pdb_lines.append(row)

# with open (pdb_file) as f:
#     for i, line in enumerate(f):
#         line = line[:-1].split()
#         if line[0] == 'ATOM':
#             if len(line) == 12: #solve issue for lines with merged columns
#                 pos = int(line[5])
#                 res = line[3]
#                 plddt = line[-2]
#                 row = [uniprot_id,file,pos,res,float(plddt)]
#                 pdb_lines.append(row)

In [ ]:
parquet_dir = "../3_outputs/processed_data/plddt_scores"
parquet_files = [f"{parquet_dir}/{par_f}" for par_f in os.listdir(parquet_dir)]
parquet_files

# 1. Get all scores for a specific protein
df = get_plddt.query_plddt(parquet_files, pdb_ids=["AF-X6R8D5-F1-model_v4"])
df

#### RSA

Calculated using script data_processing_scripts/calculate_rsa_scores.py 

In [ ]:
parquet_files = [par_f for par_f in glob.glob("../3_outputs/processed_data/rsa_scores")]
parquet_files

pdb1 = "AF-X6R8D5-F1-model_v4"
# pdb1 = "AF-Q5VUB5-F1-model_v4"

df_6lzg = get_rsa.get_structure_rsa(parquet_files, pdb1)
print(f"Retrieved {len(df_6lzg)} residues for {pdb1}")
df_6lzg

# 2. Get RSA for a specific residue
# rsa = get_rsa.get_residue_rsa(parquet_files, pdb1, "A", 124)
# rsa = get_rsa.get_residue_rsa(parquet_files, pdb1, "A", 124)
# print(f"RSA for {pdb1} chain A residue 42: {rsa}%")

# # 3. Advanced query - all residues with RSA > 50%
# all_data = get_rsa.query_rsa_data(parquet_files)
# exposed_residues = all_data[all_data['rsa'] > 50.0]
# print(f"Found {len(exposed_residues)} exposed residues (RSA > 50%)")

### Long protein sequences problem

Some proteins have very long sequences, where their structures could not be inferred by AF2 directly as a whole, and EMBL breaks their sequences into chunks for structure inferences separately.

In [2]:
# PLDDT_DIR = "../3_outputs/processed_data/plddt_scores"
# RSA_DIR = "../3_outputs/processed_data/rsa_scores"

# # Get parquet files for pLDDT and RSA scores
# parquet_files_plddt = [f"{PLDDT_DIR}/{par_f}" for par_f in os.listdir(PLDDT_DIR) if par_f.endswith('.parquet')]
# parquet_files_rsa = [f"{RSA_DIR}/{par_f}" for par_f in os.listdir(RSA_DIR) if par_f.endswith('.parquet')]

In [ ]:
pdb_id = "Q8WZ42"
df = get_plddt.query_plddt(parquet_files_plddt, pdb_ids=[f"AF-{pdb_id}-F2-model_v4"])

# df[df["residue_num"]==311]
# df[df["residue_num"]<=311].tail(10)

df = get_plddt.query_plddt(parquet_files_plddt, pdb_ids=[f"AF-{pdb_id}-F1-model_v4"])

In [ ]:
# import polars as pl
# import re
# from typing import Optional, Tuple


def find_af_file(parquet_path: str, uniprot_id: str, residue_pos: int) -> Optional[Tuple[str, int]]:
    """
    Find corresponding AF file and subfile index for a residue position
    [Uses binary search on sorted intervals for O(log n) performance]
    """
    # Load only relevant partitions
    df = pl.scan_parquet(parquet_path)
    
    result = df.filter(
        (pl.col("uniprot_id") == uniprot_id) &
        (pl.col("first_res_idx") <= residue_pos) &
        (pl.col("last_res_idx") >= residue_pos)
    ).collect()
    
    if result.height == 1:
        row = result.row(0, named=True)
        return (row["af_db_identifier"], row["subfile_index"])
    
    print(result)
    return (None, None)


def find_af_entry(parquet_path: str, uniprot_id: str, residue_pos: int) -> Optional[dict]:
    """
    Find complete AF file entry with residue range checking
    Returns full row as dictionary if found
    """
    df = pl.scan_parquet(parquet_path)
    result = df.filter(
        (pl.col("uniprot_id") == uniprot_id) &
        (pl.col("first_res_idx") <= residue_pos) &
        (pl.col("last_res_idx") >= residue_pos)
    ).collect()
    
    return result.row(0, named=True) if result.height == 1 else None


def calculate_local_position(af_entry: dict, residue_pos: int) -> int:
    """
    Calculate position within the subfile using 1-based indexing
    Formula: local_pos = residue_pos - first_res_idx + 1
    """
    return residue_pos - af_entry["first_res_idx"] + 1


# Integrated usage example
def get_af_file_and_position(parquet_path: str, uniprot_id: str, residue_pos: int):
    entry = find_af_entry(parquet_path, uniprot_id, residue_pos)
    if not entry:
        return None
        
    return {
        "af_id": entry["af_db_identifier"],
        "subfile_index": entry["subfile_index"],
        "local_position": calculate_local_position(entry, residue_pos),
        "global_range": (entry["first_res_idx"], entry["last_res_idx"])
    }


def get_plddt_score(pdb_id: str, aa_pos: int, aaref: str, 
                    parquet_files_plddt: str, accession_parquet: str, 
                    plddt_module) -> float:
    """Enhanced pLDDT lookup with dynamic AF model selection and position conversion"""
    # 1. Find correct AlphaFold file and local position
    af_entry = find_af_entry(accession_parquet, pdb_id, aa_pos)
    
    if not af_entry:
        return None
    
    # 2. Convert global position to subfile-local index
    local_pos = aa_pos - af_entry["first_res_idx"] + 1  # [1](@ref)
    
    # 3. Query pLDDT data from correct AF model
    df = plddt_module.query_plddt(
        parquet_files_plddt, 
        pdb_ids=[af_entry["af_db_identifier"]]
    )
    # 4. Validate and return score
    try:
        if (local_pos in df.index and 
            df.loc[local_pos, "residue_name"] == aaref):
            return df.loc[local_pos, "plddt"]
    except (KeyError, TypeError):
        pass
    
    return None


# Usage example

# print(find_af_file("../3_outputs/processed_data/af_id_mapping/af_mapping.parquet", 
#                    "A0A2I1YHU5", 500))
# Output: ("AF-A0A2I1YHU5-F1", 1)

# find_af_file("../3_outputs/processed_data/af_id_mapping/af_mapping.parquet", pdb_id, 1400)

## 4. gnomad

Allele frequency thresholds:
1 / 100,000

In [ ]:
## Define some constants
VCF_DIR = "../1_inputs/raw_inputs/gnomad"
CHROMOSOMES = [str(c) for c in list(range(1, 23))] + ['X', 'Y']

# VARIANT_LIST = f"../3_outputs/processed_data/gnomad_allele_frequencies/gnomad_variants_list.txt"
# gene_info_pd[["chrom","chr_pos_38","ref","alt"]].to_csv(VARIANT_LIST, sep="\t", header=False, index=False)

##### Sort out all the vcf files and all their position coordinates for looking up varchamp-related variants

In [ ]:
# import subprocess

# Set strict error handling
# def run_command(cmd):
#     result = subprocess.run(cmd, shell=True, check=True, text=True)
#     return result

# # !conda install bcftools -c conda-forge -y ## install bcftools if you didn't

# OUTPUT_DIR = f"../3_outputs/processed_data/gnomad_allele_frequencies"
# os.makedirs(OUTPUT_DIR, exist_ok=True)

# # Process chromosomes
# for chrom in CHROMOSOMES:
#     vcf_file = f"{VCF_DIR}/gnomad.exomes.v4.1.sites.chr{chrom}.vcf.bgz"
#     csi_file = f"{vcf_file}.csi"
#     output_file = f"{OUTPUT_DIR}/chr{chrom}_allele_frequencies.txt"
    
#     if os.path.exists(vcf_file):
#         print(f"Processing chromosome {chrom}...")
        
#         if not os.path.exists(csi_file):
#             print(f"Indexing {vcf_file}")
#             run_command(f"bcftools index --threads 4 {vcf_file}")
        
#         print(f"Extracting frequencies for {chrom}")
#         view_cmd = f"bcftools view -R {VARIANT_LIST} {vcf_file}"
#         query_cmd = "bcftools query -f '%CHROM\\t%POS\\t%REF\\t%ALT\\t%AF\\n'"
        
#         with open(output_file, 'w') as f:
#             p1 = subprocess.Popen(view_cmd.split(), stdout=subprocess.PIPE)
#             p2 = subprocess.Popen(query_cmd.split(), stdin=p1.stdout, stdout=f)
#             p1.stdout.close()
#             p2.communicate()
            
#         print(f"Saved to {output_file}")
#     else:
#         print(f"Skipping chromosome {chrom} - file not found")

# print("Processing completed")

#### Merge the gnomad AF files from chromosomes

In [ ]:
allele_freq_dfs = []
for chrom in CHROMOSOMES:
    try:
        allele_freq = pl.read_csv(f"../3_outputs/processed_data/gnomad_allele_frequencies/chr{chrom}_allele_frequencies.txt",
                                separator="\t", has_header=False, 
                                new_columns=["chr","chr_pos_38","ref","alt","allele_freq"])
        allele_freq = allele_freq.with_columns(
            pl.col("chr").str.replace(r"\'", "").str.replace(r"\'", "").str.replace("chr", "")
        ).drop_nulls()
        # allele_freq = allele_freq.filter(~pl.all_horizontal(pl.all().is_null()))
        # print(allele_freq)
        allele_freq_dfs.append(allele_freq)
    except:
        continue

allele_freq_all = pl.concat(allele_freq_dfs)
allele_freq_gnomad = allele_freq_all.with_columns(
    pl.concat_str([pl.col("chr"),
                   pl.lit('-'),
                   pl.col("chr_pos_38").cast(pl.String),
                   pl.lit('-'),
                   pl.col("ref"),
                   pl.lit('-'),
                   pl.col("alt")
                   ]).alias("varid"),
    pl.col("allele_freq").alias("gnomad_af")
).drop(pl.col("allele_freq"))#.to_pandas()

allele_freq_gnomad